In [1]:
# import sys
# !{sys.executable} -m pip install --upgrade \
#     datasets \
#     evaluate \
#     transformers \
#     accelerate \
#     torch \
#     numpy \
#     scikit-learn

In [2]:
# import sys
# !{sys.executable} -m pip install evaluate

In [3]:
# import sys
# !{sys.executable} -m pip install scipy scikit-learn

In [4]:
# import sys
# !{sys.executable} -m pip install "accelerate>=1.1.0" --upgrade

In [5]:
!pip list

Package         Version
--------------- --------
accelerate      1.13.0
annotated-doc   0.0.4
click           8.4.1
datasets        4.8.5
dill            0.4.1
evaluate        0.4.6
filelock        3.29.0
fsspec          2026.2.0
hf-xet          1.5.0
huggingface_hub 1.17.0
joblib          1.5.3
markdown-it-py  4.2.0
mdurl           0.1.2
multiprocess    0.70.19
pyarrow         24.0.0
regex           2026.5.9
rich            15.0.0
safetensors     0.7.0
scikit-learn    1.7.2
scipy           1.15.3
shellingham     1.5.4
threadpoolctl   3.6.0
tokenizers      0.22.2
tqdm            4.67.3
transformers    5.9.0
typer           0.25.1
xxhash          3.7.0


In [6]:
import torch

In [7]:
import os

# ── WAJIB di set SEBELUM import torch ────────────────────────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-cd3e3bb9-ccf3-5d52-9aca-0ac3947ab62e"

import time
import torch
import threading
import csv
from datetime import datetime
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# ── Verify GPU ────────────────────────────────────────────────────────────────
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device name    : {torch.cuda.get_device_name(0)}")
print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ── VRAM Logger + Real-Time Monitor (background thread) ──────────────────────
log_file = "vram_log.csv"
logging_active = True
vram_history = []  # untuk kalkulasi rata-rata di akhir

def vram_logger(interval=1.0):
    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "timestamp",
            "allocated_gb",   # tensor aktif
            "reserved_gb",    # cache PyTorch
            "cache_overhead_gb",  # reserved - allocated
            "free_gb",        # sisa VRAM tersedia
            "total_gb",
            "allocated_pct",
            "reserved_pct",
        ])
        while logging_active:
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved  = torch.cuda.memory_reserved()  / 1e9
            total     = torch.cuda.get_device_properties(0).total_memory / 1e9
            free      = total - reserved
            overhead  = reserved - allocated
            alloc_pct = allocated / total * 100
            resv_pct  = reserved  / total * 100

            vram_history.append(allocated)

            writer.writerow([
                datetime.now().isoformat(),
                f"{allocated:.3f}",
                f"{reserved:.3f}",
                f"{overhead:.3f}",
                f"{free:.3f}",
                f"{total:.2f}",
                f"{alloc_pct:.1f}",
                f"{resv_pct:.1f}",
            ])
            f.flush()

            ts = datetime.now().strftime("%H:%M:%S")
            print(
                f"\r[{ts}] "
                f"Model(Alloc): {allocated:.3f} GB ({alloc_pct:.1f}%) | "
                f"Cache(Resv): {reserved:.3f} GB ({resv_pct:.1f}%) | "
                f"Overhead: {overhead:.3f} GB | "
                f"Free: {free:.3f} GB | "
                f"Total: {total:.2f} GB",
                end="", flush=True
            )
            time.sleep(interval)

logger_thread = threading.Thread(target=vram_logger, daemon=True)
logger_thread.start()

# ── Dataset & Tokenizer ───────────────────────────────────────────────────────
print("\nLoading dataset...")
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
checkpoint   = "bert-base-uncased"
tokenizer    = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)

tokenized = raw_datasets.map(tokenize_fn, batched=True)
model     = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# ── Verify model ada di GPU ───────────────────────────────────────────────────
model = model.to("cuda")
print(f"\nModel device   : {next(model.parameters()).device}")

# ── Metric ────────────────────────────────────────────────────────────────────
metric = evaluate.load("glue", "mrpc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

# ── Training Args ─────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# ── Benchmark Training ────────────────────────────────────────────────────────
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n{'='*65}")
print(f"  Benchmark Start : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Model           : {checkpoint}")
print(f"  GPU             : {torch.cuda.get_device_name(0)}")
print(f"  VRAM total      : {total_vram:.2f} GB")
print(f"{'='*65}\n")

t_start = time.time()
trainer.train()
t_end   = time.time()

# ── Stop Logger ───────────────────────────────────────────────────────────────
logging_active = False
print()  # newline setelah \r monitor

# ── Summary ───────────────────────────────────────────────────────────────────
peak_alloc   = torch.cuda.max_memory_allocated() / 1e9
peak_resv    = torch.cuda.max_memory_reserved()  / 1e9
avg_alloc    = sum(vram_history) / len(vram_history) if vram_history else 0
duration     = t_end - t_start
total_vram   = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\n{'='*65}")
print(f"  {'BENCHMARK SUMMARY':^59}")
print(f"{'='*65}")
print(f"  Training time     : {duration/60:.2f} menit ({duration:.1f} detik)")
print(f"{'─'*65}")
print(f"  {'VRAM — Allocated (tensor aktif)':}")
print(f"    Peak            : {peak_alloc:.3f} GB  ({peak_alloc/total_vram*100:.1f}% of {total_vram:.2f} GB)")
print(f"    Average         : {avg_alloc:.3f} GB  ({avg_alloc/total_vram*100:.1f}% of {total_vram:.2f} GB)")
print(f"  {'VRAM — Reserved (PyTorch cache)':}")
print(f"    Peak            : {peak_resv:.3f} GB  ({peak_resv/total_vram*100:.1f}% of {total_vram:.2f} GB)")
print(f"    Cache overhead  : {peak_resv - peak_alloc:.3f} GB  (reserved - allocated)")
print(f"  {'VRAM — Headroom':}")
print(f"    Sisa (dari peak): {total_vram - peak_resv:.3f} GB")
print(f"{'─'*65}")
print(f"  Log saved         : {log_file}")
print(f"{'='*65}")

/home/atmind/work/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available : True
Device name    : NVIDIA H200 MIG 2g.35gb
VRAM total     : 34.90 GB

Loading dataset...
[12:19:53] Model(Alloc): 0.000 GB (0.0%) | Cache(Resv): 0.000 GB (0.0%) | Overhead: 0.000 GB | Free: 34.897 GB | Total: 34.90 GB

[12:19:57] Model(Alloc): 0.000 GB (0.0%) | Cache(Resv): 0.000 GB (0.0%) | Overhead: 0.000 GB | Free: 34.897 GB | Total: 34.90 GB

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10375.62it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo


Model device   : cuda:0
[12:20:00] Model(Alloc): 0.439 GB (1.3%) | Cache(Resv): 0.495 GB (1.4%) | Overhead: 0.056 GB | Free: 34.402 GB | Total: 34.90 GB
  Benchmark Start : 2026-05-29 12:20:00
  Model           : bert-base-uncased
  GPU             : NVIDIA H200 MIG 2g.35gb
  VRAM total      : 34.90 GB



...

[12:20:32] Model(Alloc): 1.821 GB (5.2%) | Cache(Resv): 2.508 GB (7.2%) | Overhead: 0.687 GB | Free: 32.388 GB | Total: 34.90 GB

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[12:20:33] Model(Alloc): 1.387 GB (4.0%) | Cache(Resv): 2.508 GB (7.2%) | Overhead: 1.121 GB | Free: 32.388 GB | Total: 34.90 GB

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


[12:20:36] Model(Alloc): 1.387 GB (4.0%) | Cache(Resv): 2.508 GB (7.2%) | Overhead: 1.121 GB | Free: 32.388 GB | Total: 34.90 GB

                       BENCHMARK SUMMARY                     
  Training time     : 0.59 menit (35.4 detik)
─────────────────────────────────────────────────────────────────
  VRAM — Allocated (tensor aktif)
    Peak            : 2.308 GB  (6.6% of 34.90 GB)
    Average         : 1.341 GB  (3.8% of 34.90 GB)
  VRAM — Reserved (PyTorch cache)
    Peak            : 2.508 GB  (7.2% of 34.90 GB)
    Cache overhead  : 0.200 GB  (reserved - allocated)
  VRAM — Headroom
    Sisa (dari peak): 32.388 GB
─────────────────────────────────────────────────────────────────
  Log saved         : vram_log.csv
